# ComfyUI — WAI Illustrious v15 + Bleach Watercolor

Notebook này chạy [ComfyUI](https://github.com/comfyanonymous/ComfyUI) trên GPU T4 miễn phí của Google Colab, gắn sẵn:

- Model nền **WAI-illustrious-SDXL v15** (`WAI-Nsfw-Illustrious-15.safetensors`, VAE đã gắn trong model)
- LoRA **Bleach Watercolor Style (Tite Kubo)** (`bleach_watercolor.safetensors`, từ kích hoạt `bleach_watercolor`, weight mặc định 0.85)

**Thứ tự trên điện thoại:** Runtime → Change runtime type → GPU **T4** → lưu. Chạy lần lượt từng ô ▶ từ trên xuống. Token Civitai chỉ dán vào ô mật khẩu trong Colab, **đừng gửi token cho ai và đừng lưu token vào notebook**.

Cách lấy token: vào [civitai.com/user/account](https://civitai.com/user/account), kéo xuống mục **API Keys** → **Add API key**, sao chép token (chỉ hiện một lần).


In [ ]:
#@title Giữ Colab không bị tắt khi để nền (nên chạy trên điện thoại)
%%html
<p>Nhấn phát đoạn im lặng này rồi để tab Colab mở khi đang tạo ảnh.</p>
<audio src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" controls loop></audio>


In [ ]:
#@title 1) Cài đặt (chạy ô này trước)
import os, re, sys
import ipywidgets as w
from IPython.display import display, HTML, Markdown

# Bộ nhớ trong phiên Colab thôi, không ghi token ra file hay in ra màn hình.
if "BLEACH_CFG" not in globals():
    BLEACH_CFG = {}

_token = w.Password(
    placeholder="Dán API token Civitai vào đây",
    layout=w.Layout(width="100%"),
)
_drive = w.Checkbox(
    value=False,
    description="Lưu model trên Google Drive (lần sau khỏi tải lại)",
    indent=False,
    layout=w.Layout(width="100%"),
)
_weight = w.BoundedFloatText(
    value=0.85,
    min=0.0,
    max=2.0,
    step=0.05,
    description="Weight LoRA:",
    style={"description_width": "110px"},
    layout=w.Layout(width="100%"),
)
_extra = w.Textarea(
    placeholder="Link LoRA Civitai thêm (không bắt buộc), mỗi dòng một link hoặc modelVersionId",
    layout=w.Layout(width="100%", height="70px"),
)
_status = w.HTML("")
_btn = w.Button(description="Lưu cài đặt", button_style="success", layout=w.Layout(width="200px", height="40px"))

def _save(_=None):
    tok = (_token.value or "").strip()
    if not tok:
        _status.value = '<p style="color:#b00020"><b>Thiếu token Civitai.</b> Dán token vào ô mật khẩu rồi bấm Lưu cài đặt.</p>'
        return
    BLEACH_CFG["token"] = tok
    BLEACH_CFG["use_drive"] = bool(_drive.value)
    BLEACH_CFG["lora_weight"] = float(_weight.value)
    extra_lines = [ln.strip() for ln in (_extra.value or "").splitlines() if ln.strip()]
    BLEACH_CFG["extra_lora_urls"] = extra_lines
    _token.value = ""  # xóa khỏi ô, tránh bị lưu vào file notebook
    _status.value = (
        f'<p style="color:#0a7a2f"><b>Đã lưu cài đặt.</b> Token được giữ trong phiên này, không in ra. '
        f'Weight LoRA = {BLEACH_CFG["lora_weight"]}. '
        f'Google Drive: {"bật" if BLEACH_CFG["use_drive"] else "tắt"}. '
        f'LoRA thêm: {len(extra_lines)}.</p>'
    )

_btn.on_click(_save)
display(HTML("""
<h3>Cài đặt</h3>
<p>Token Civitai (ô ẩn ký tự, chỉ nhập trong Colab):</p>
"""))
display(_token, _drive, _weight, HTML("<p>LoRA thêm (không bắt buộc):</p>"), _extra, _btn, _status)
display(Markdown("Khuyên dùng weight **0.7–1.0** (mặc định 0.85). Sau khi lưu, chạy ô **2) Cài đặt và khởi động ComfyUI**." ))


In [ ]:
#@title 2) Cài đặt ComfyUI, tải model và mở link
import os, re, sys, json, time, shutil, stat, tempfile, subprocess, threading, urllib.parse
from pathlib import Path
from IPython.display import display, HTML, Markdown, FileLink

def _die(msg):
    display(HTML(f'<p style="color:#b00020;font-size:1.05rem"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "BLEACH_CFG" not in globals() or not BLEACH_CFG.get("token"):
    _die("Bạn chưa lưu token Civitai. Hãy chạy ô 1) Cài đặt, dán token, bấm Lưu cài đặt, rồi chạy lại ô này.")

TOKEN = BLEACH_CFG["token"]
USE_DRIVE = bool(BLEACH_CFG.get("use_drive"))
LORA_WEIGHT = float(BLEACH_CFG.get("lora_weight", 0.85))
EXTRA_URLS = list(BLEACH_CFG.get("extra_lora_urls") or [])

CKPT_VERSION = 2167369
LORA_VERSION = 2744203
CKPT_NAME = "WAI-Nsfw-Illustrious-15.safetensors"
LORA_NAME = "bleach_watercolor.safetensors"
CKPT_MIN = 6_000_000_000
LORA_MIN = 50_000_000

IN_COLAB = Path("/content").exists()
ROOT = Path("/content" if IN_COLAB else "/tmp/comfy_runtime")
COMFY = ROOT / "ComfyUI"
DRIVE_ROOT = Path("/content/drive/MyDrive/rpchat_comfyui")

# ---- Google Drive (tùy chọn) ----
if USE_DRIVE:
    if not IN_COLAB:
        _die("Google Drive chỉ dùng được khi chạy trên Google Colab.")
    try:
        from google.colab import drive
        display(Markdown("Đang gắn Google Drive…"))
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        _die("Không gắn được Google Drive. Bỏ tick ô Drive ở bước 1 rồi chạy lại, hoặc cho phép Colab truy cập Drive.")
    (DRIVE_ROOT / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / "models" / "loras").mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / "output").mkdir(parents=True, exist_ok=True)

# ---- Cài gói cần thiết, không đụng bản PyTorch sẵn có trên Colab ----
display(Markdown("Đang cài công cụ tải và sao chép ComfyUI…"))
subprocess.run(["apt-get", "update", "-qq"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "wget"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not (COMFY / "main.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY)], check=True)

req_path = COMFY / "requirements.txt"
skip = {"torch", "torchvision", "torchaudio"}
filtered = []
for line in req_path.read_text().splitlines():
    raw = line.strip()
    if not raw or raw.startswith("#"):
        continue
    name = re.split(r"[=<>\[]", raw, maxsplit=1)[0].strip().lower()
    if name not in skip:
        filtered.append(raw)
if filtered:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *filtered], check=True)

# ---- Thư mục model (symlink sang Drive nếu bật) ----
ckpt_dir = COMFY / "models" / "checkpoints"
lora_dir = COMFY / "models" / "loras"
ckpt_dir.mkdir(parents=True, exist_ok=True)
lora_dir.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    def _link(src: Path, dest: Path):
        if dest.is_symlink() or dest.exists():
            if dest.is_dir() and not dest.is_symlink():
                # chuyển file đã có (nếu có) sang Drive rồi thay bằng symlink
                for p in dest.glob("*"):
                    target = src / p.name
                    if not target.exists():
                        shutil.move(str(p), str(target))
                shutil.rmtree(dest)
            else:
                dest.unlink()
        dest.symlink_to(src, target_is_directory=True)
    _link(DRIVE_ROOT / "models" / "checkpoints", ckpt_dir)
    _link(DRIVE_ROOT / "models" / "loras", lora_dir)

ckpt_path = ckpt_dir / CKPT_NAME
lora_path = lora_dir / LORA_NAME

# ---- Tải từ Civitai: token chỉ nằm trong file tạm 0600, không in, không đưa vào argv ----
def _civitai_ok(version_id: int, token: str) -> tuple[bool, str]:
    """Kiểm tra endpoint tải, không in URL hay token."""
    import urllib.request
    url = f"https://civitai.com/api/download/models/{version_id}"
    req = urllib.request.Request(url, method="GET", headers={
        "Authorization": f"Bearer {token}",
        "User-Agent": "rpchat-comfyui-colab",
    })
    try:
        opener = urllib.request.build_opener(urllib.request.HTTPRedirectHandler)
        # Chỉ lấy header: dùng method GET nhưng ngắt sau khi có redirect/status
        class NoRedirect(urllib.request.HTTPRedirectHandler):
            def redirect_request(self, req, fp, code, msg, headers, newurl):
                return None
        opener = urllib.request.build_opener(NoRedirect)
        try:
            resp = opener.open(req, timeout=45)
            code = getattr(resp, "status", 200)
            loc = ""
            resp.close()
        except urllib.error.HTTPError as e:
            code = e.code
            loc = e.headers.get("Location") or ""
        if code in (200, 206):
            return True, ""
        if code in (301, 302, 303, 307, 308):
            low = loc.lower()
            if "login" in low or "reason=download-auth" in low:
                return False, "invalid"
            if loc:
                return True, ""
        if code in (401, 403):
            return False, "invalid"
        return False, f"http_{code}"
    except Exception:
        return False, "network"

def _download_civitai(version_id: int, dest: Path, token: str, min_bytes: int, label: str):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        mb = dest.stat().st_size / 1024 / 1024
        display(Markdown(f"Đã có **{dest.name}** ({mb:.1f} MB), bỏ qua tải."))
        return
    ok, why = _civitai_ok(version_id, token)
    if not ok:
        if why == "invalid":
            _die("Token Civitai thiếu hoặc không hợp lệ, hoặc không đủ quyền tải file này. Kiểm tra lại token ở civitai.com/user/account rồi lưu ở ô 1.")
        if why == "network":
            _die("Không kết nối được tới Civitai. Thử chạy lại ô này sau.")
        _die(f"Civitai từ chối tải {label} (mã {why}). Thử lại sau hoặc kiểm tra token.")
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    display(Markdown(f"Đang tải **{label}** về `{dest.name}`… Có thể mất vài phút."))
    cfg = Path(tempfile.mkstemp(prefix="civitai_", suffix=".cfg")[1])
    try:
        os.chmod(cfg, 0o600)
        # Đúng endpoint Civitai; token đặt trong file cấu hình, không nằm trên dòng lệnh.
        cfg.write_text(
            "location\n"
            "fail\n"
            "silent\n"
            "show-error\n"
            "continue-at -\n"
            f"output = \"{part}\"\n"
            f"url = \"https://civitai.com/api/download/models/{version_id}?token={token}\"\n"
        )
        os.chmod(cfg, 0o600)
        r = subprocess.run(["curl", "--config", str(cfg)], capture_output=True)
        if r.returncode != 0 or not part.exists() or part.stat().st_size < min_bytes:
            # Thử aria2c nếu curl lỗi
            if shutil.which("aria2c"):
                inp = Path(tempfile.mkstemp(prefix="aria_", suffix=".txt")[1])
                try:
                    os.chmod(inp, 0o600)
                    inp.write_text(f"https://civitai.com/api/download/models/{version_id}?token={token}\n out={part.name}\n dir={part.parent}\n")
                    os.chmod(inp, 0o600)
                    r2 = subprocess.run(
                        ["aria2c", "-c", "-x", "16", "-s", "16", "--console-log-level=error", "-i", str(inp)],
                        capture_output=True,
                    )
                    if r2.returncode != 0:
                        _die("Tải file thất bại. Token có thể không hợp lệ, hoặc mạng bị ngắt. Thử chạy lại ô này.")
                finally:
                    try:
                        inp.unlink(missing_ok=True)
                    except Exception:
                        pass
            else:
                _die("Tải file thất bại. Token có thể không hợp lệ, hoặc mạng bị ngắt. Thử chạy lại ô này.")
        if not part.exists() or part.stat().st_size < min_bytes:
            if part.exists():
                part.unlink()
            _die(f"File {label} tải không đủ dung lượng. Xóa file dở rồi chạy lại ô này. Kiểm tra token nếu lỗi lặp lại.")
        part.replace(dest)
        display(Markdown(f"Đã tải xong **{dest.name}** ({dest.stat().st_size/1024/1024:.1f} MB)."))
    finally:
        try:
            cfg.unlink(missing_ok=True)
        except Exception:
            pass

def _parse_version_id(text: str):
    text = text.strip()
    m = re.search(r"modelVersionId=(\d+)", text)
    if m:
        return int(m.group(1))
    m = re.search(r"/api/download/models/(\d+)", text)
    if m:
        return int(m.group(1))
    if re.fullmatch(r"\d+", text):
        return int(text)
    m = re.search(r"civitai\.com/models/(\d+)", text)
    if m:
        # Không có version id: lấy bản mới nhất qua API công khai (không cần token)
        import urllib.request
        mid = m.group(1)
        try:
            with urllib.request.urlopen(f"https://civitai.com/api/v1/models/{mid}", timeout=45) as resp:
                data = json.loads(resp.read().decode())
            vers = data.get("modelVersions") or []
            if vers:
                return int(vers[0]["id"])
        except Exception:
            return None
    return None

_download_civitai(CKPT_VERSION, ckpt_path, TOKEN, CKPT_MIN, "model nền WAI-illustrious v15 (~6,5 GB)")
_download_civitai(LORA_VERSION, lora_path, TOKEN, LORA_MIN, "LoRA Bleach Watercolor (~109 MB)")

extra_files = []
for i, raw in enumerate(EXTRA_URLS, start=1):
    vid = _parse_version_id(raw)
    if not vid:
        display(HTML(f'<p style="color:#b00020">Bỏ qua LoRA thêm không đọc được: (dòng {i}). Cần link Civitai có modelVersionId hoặc id phiên bản.</p>'))
        continue
    fname = f"extra_lora_{i}.safetensors"
    dest = lora_dir / fname
    _download_civitai(vid, dest, TOKEN, 1_000_000, f"LoRA thêm #{i}")
    extra_files.append(fname)

BLEACH_CFG["extra_lora_files"] = extra_files

# ---- Workflow mặc định ----
POS = (
    "masterpiece, best quality, amazing quality, official art, traditional media, "
    "watercolor (medium), bleach_watercolor, 1girl, solo, inoue orihime, bleach, "
    "long hair, orange hair, brown eyes, parted bangs, hair ornament, hair flower, "
    "blue flower, green shirt, frilled shirt, short sleeves, blue jeans, "
    "white background, simple background, cowboy shot"
)
NEG = (
    "bad quality, worst quality, worst detail, nsfw, explicit, monochrome, "
    "greyscale, 3d, realistic, english text, watermark, signature, censor"
)
WIDTH, HEIGHT, STEPS, CFG = 832, 1216, 28, 6.0

def build_ui_workflow(weight=LORA_WEIGHT):
    return {
        "last_node_id": 8, "last_link_id": 11,
        "nodes": [
            {"id": 1, "type": "CheckpointLoaderSimple", "pos": [40, 200], "size": [360, 98],
             "flags": {}, "order": 0, "mode": 0, "inputs": [],
             "outputs": [
                 {"name": "MODEL", "type": "MODEL", "links": [1], "slot_index": 0},
                 {"name": "CLIP", "type": "CLIP", "links": [2], "slot_index": 1},
                 {"name": "VAE", "type": "VAE", "links": [10], "slot_index": 2}],
             "properties": {"Node name for S&R": "CheckpointLoaderSimple"},
             "widgets_values": [CKPT_NAME]},
            {"id": 2, "type": "LoraLoader", "pos": [430, 200], "size": [340, 126],
             "flags": {}, "order": 1, "mode": 0,
             "inputs": [{"name": "model", "type": "MODEL", "link": 1},
                        {"name": "clip", "type": "CLIP", "link": 2}],
             "outputs": [
                 {"name": "MODEL", "type": "MODEL", "links": [3], "slot_index": 0},
                 {"name": "CLIP", "type": "CLIP", "links": [4, 5], "slot_index": 1}],
             "properties": {"Node name for S&R": "LoraLoader"},
             "widgets_values": [LORA_NAME, float(weight), float(weight)]},
            {"id": 3, "type": "CLIPTextEncode", "pos": [800, 40], "size": [400, 200],
             "flags": {}, "order": 2, "mode": 0, "title": "Positive",
             "inputs": [{"name": "clip", "type": "CLIP", "link": 4}],
             "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [6], "slot_index": 0}],
             "properties": {"Node name for S&R": "CLIPTextEncode"},
             "widgets_values": [POS]},
            {"id": 4, "type": "CLIPTextEncode", "pos": [800, 280], "size": [400, 160],
             "flags": {}, "order": 3, "mode": 0, "title": "Negative",
             "inputs": [{"name": "clip", "type": "CLIP", "link": 5}],
             "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [7], "slot_index": 0}],
             "properties": {"Node name for S&R": "CLIPTextEncode"},
             "widgets_values": [NEG]},
            {"id": 5, "type": "EmptyLatentImage", "pos": [800, 500], "size": [280, 106],
             "flags": {}, "order": 4, "mode": 0, "inputs": [],
             "outputs": [{"name": "LATENT", "type": "LATENT", "links": [8], "slot_index": 0}],
             "properties": {"Node name for S&R": "EmptyLatentImage"},
             "widgets_values": [WIDTH, HEIGHT, 1]},
            {"id": 6, "type": "KSampler", "pos": [1240, 200], "size": [300, 262],
             "flags": {}, "order": 5, "mode": 0,
             "inputs": [{"name": "model", "type": "MODEL", "link": 3},
                        {"name": "positive", "type": "CONDITIONING", "link": 6},
                        {"name": "negative", "type": "CONDITIONING", "link": 7},
                        {"name": "latent_image", "type": "LATENT", "link": 8}],
             "outputs": [{"name": "LATENT", "type": "LATENT", "links": [9], "slot_index": 0}],
             "properties": {"Node name for S&R": "KSampler"},
             "widgets_values": [0, "randomize", STEPS, CFG, "euler_ancestral", "normal", 1.0]},
            {"id": 7, "type": "VAEDecode", "pos": [1580, 200], "size": [240, 46],
             "flags": {}, "order": 6, "mode": 0,
             "inputs": [{"name": "samples", "type": "LATENT", "link": 9},
                        {"name": "vae", "type": "VAE", "link": 10}],
             "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [11], "slot_index": 0}],
             "properties": {"Node name for S&R": "VAEDecode"}, "widgets_values": []},
            {"id": 8, "type": "SaveImage", "pos": [1860, 200], "size": [300, 260],
             "flags": {}, "order": 7, "mode": 0,
             "inputs": [{"name": "images", "type": "IMAGE", "link": 11}],
             "outputs": [{"name": "images", "type": "IMAGE", "links": None, "slot_index": 0}],
             "properties": {"Node name for S&R": "SaveImage"},
             "widgets_values": ["bleach_orihime"]},
        ],
        "links": [
            [1, 1, 0, 2, 0, "MODEL"], [2, 1, 1, 2, 1, "CLIP"], [3, 2, 0, 6, 0, "MODEL"],
            [4, 2, 1, 3, 0, "CLIP"], [5, 2, 1, 4, 0, "CLIP"], [6, 3, 0, 6, 1, "CONDITIONING"],
            [7, 4, 0, 6, 2, "CONDITIONING"], [8, 5, 0, 6, 3, "LATENT"], [9, 6, 0, 7, 0, "LATENT"],
            [10, 1, 2, 7, 1, "VAE"], [11, 7, 0, 8, 0, "IMAGE"],
        ],
        "groups": [], "config": {},
        "extra": {"ds": {"scale": 0.85, "offset": [0, 0]}, "workflow_name": "bleach_orihime"},
        "version": 0.4,
    }

def build_api_workflow(weight=LORA_WEIGHT, seed=0, width=WIDTH, height=HEIGHT, steps=STEPS,
                       cfg=CFG, batch=1, pos=POS, neg=NEG, extra=None):
    last = "10"
    g = {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": CKPT_NAME}},
        "10": {"class_type": "LoraLoader", "inputs": {
            "model": ["1", 0], "clip": ["1", 1], "lora_name": LORA_NAME,
            "strength_model": float(weight), "strength_clip": float(weight)}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {
            "width": int(width), "height": int(height), "batch_size": int(batch)}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": pos, "clip": [last, 1]}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"text": neg, "clip": [last, 1]}},
        "3": {"class_type": "KSampler", "inputs": {
            "seed": int(seed), "steps": int(steps), "cfg": float(cfg),
            "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0,
            "model": [last, 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["1", 2]}},
        "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": "bleach_orihime", "images": ["8", 0]}},
    }
    prev = "10"
    for i, fname in enumerate(extra or []):
        nid = str(20 + i)
        g[nid] = {"class_type": "LoraLoader", "inputs": {
            "model": [prev, 0], "clip": [prev, 1], "lora_name": fname,
            "strength_model": float(weight), "strength_clip": float(weight)}}
        prev = nid
    if prev != "10":
        g["6"]["inputs"]["clip"] = [prev, 1]
        g["7"]["inputs"]["clip"] = [prev, 1]
        g["3"]["inputs"]["model"] = [prev, 0]
    return g

ui = build_ui_workflow(LORA_WEIGHT)
api = build_api_workflow(LORA_WEIGHT, extra=extra_files)
(COMFY / "default_workflow.json").write_text(json.dumps(ui, ensure_ascii=False, indent=2))
wf_dir = COMFY / "user" / "default" / "workflows"
wf_dir.mkdir(parents=True, exist_ok=True)
(wf_dir / "bleach_orihime.json").write_text(json.dumps(ui, ensure_ascii=False, indent=2))
out_json = ROOT / "bleach_orihime_workflow.json"
out_json.write_text(json.dumps(ui, ensure_ascii=False, indent=2))
(ROOT / "bleach_orihime_api.json").write_text(json.dumps(api, ensure_ascii=False, indent=2))
BLEACH_CFG["comfy"] = str(COMFY)
BLEACH_CFG["pos"] = POS
BLEACH_CFG["neg"] = NEG
display(Markdown("Đã ghi workflow mặc định (ComfyUI sẽ mở sẵn khi vào giao diện). File JSON để tải:" ))
try:
    from google.colab import files as colab_files
    # Chỉ cho tải file workflow, không đụng tới token
    display(HTML(f"<p>File workflow: <code>{out_json}</code></p>"))
except Exception:
    display(Markdown(f"File workflow: `{out_json}`"))

# ---- cloudflared ----
cf_bin = ROOT / "cloudflared"
if not cf_bin.exists():
    subprocess.run([
        "wget", "-q", "-O", str(cf_bin),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ], check=True)
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

# ---- Khởi động ComfyUI ----
# Tắt process cũ nếu chạy lại ô này
for proc_name in ("main.py", "cloudflared"):
    subprocess.run(["pkill", "-f", f"{COMFY}.*main.py" if proc_name == "main.py" else str(cf_bin)], check=False)

log_path = ROOT / "comfyui.log"
cf_log = ROOT / "cloudflared.log"
log_f = open(log_path, "w")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
     "--preview-method", "auto", "--enable-cors-header"],
    cwd=str(COMFY),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env={**os.environ, "HF_HUB_DISABLE_TELEMETRY": "1"},
)
BLEACH_CFG["comfy_proc"] = comfy_proc

import urllib.request
ready = False
for _ in range(90):
    if comfy_proc.poll() is not None:
        tail = log_path.read_text(errors="ignore")[-2000:]
        _die("ComfyUI thoát sớm. Một đoạn log (không chứa token): " + tail.replace("<", " "))
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(2)
if not ready:
    _die("ComfyUI không sẵn sàng sau khoảng 3 phút. Chạy lại ô này, hoặc xem runtime đã chọn GPU T4 chưa.")

cf_f = open(cf_log, "w")
cf_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"],
    stdout=cf_f,
    stderr=subprocess.STDOUT,
)
BLEACH_CFG["cf_proc"] = cf_proc

url = None
for _ in range(40):
    txt = cf_log.read_text(errors="ignore") if cf_log.exists() else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break
    time.sleep(1)
if not url:
    _die("Chưa lấy được link trycloudflare. Chạy lại ô này. Nếu Colab hỏi cấp quyền mạng thì hãy cho phép.")

BLEACH_CFG["public_url"] = url
display(HTML(f"""
<div style="border:2px solid #0a7a2f;padding:16px;border-radius:12px;margin:12px 0">
  <p style="margin:0 0 8px;font-size:1.1rem"><b>ComfyUI đã sẵn sàng.</b> Mở link này trên điện thoại:</p>
  <p style="margin:0;font-size:1.2rem"><a href="{url}" target="_blank">{url}</a></p>
  <p style="margin:8px 0 0;color:#333">Workflow Orihime / Bleach Watercolor được mở sẵn. Bấm <b>Queue</b> để ra ảnh. Nếu không thấy, vào mục Workflows chọn <code>bleach_orihime</code>.</p>
</div>
"""))
display(Markdown("Muốn tạo ảnh ngay trong Colab, kéo xuống ô **Chế độ đơn giản** rồi bấm tạo." ))


## 3) Chế độ đơn giản (dùng trên điện thoại, không cần vào giao diện ComfyUI)

Chạy ô dưới sau khi ô 2 đã hiện link. Ảnh hiện ngay trong notebook; nếu bạn bật Google Drive thì ảnh cũng được lưu vào `MyDrive/rpchat_comfyui/output`.


In [ ]:
#@title 3) Chế độ đơn giản — gõ prompt và tạo ảnh
import os, json, time, random, shutil, urllib.request, urllib.error
from pathlib import Path
from IPython.display import display, HTML, Markdown, Image as IPyImage
import ipywidgets as w

def _die(msg):
    display(HTML(f'<p style="color:#b00020"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "BLEACH_CFG" not in globals() or not BLEACH_CFG.get("comfy"):
    _die("Chưa khởi động ComfyUI. Chạy ô 1 rồi ô 2 trước.")

POS0 = BLEACH_CFG.get("pos") or (
    "masterpiece, best quality, amazing quality, official art, traditional media, "
    "watercolor (medium), bleach_watercolor, 1girl, solo, inoue orihime, bleach, "
    "long hair, orange hair, brown eyes, parted bangs, hair ornament, hair flower, "
    "blue flower, green shirt, frilled shirt, short sleeves, blue jeans, "
    "white background, simple background, cowboy shot"
)
NEG0 = BLEACH_CFG.get("neg") or (
    "bad quality, worst quality, worst detail, nsfw, explicit, monochrome, "
    "greyscale, 3d, realistic, english text, watermark, signature, censor"
)
WEIGHT = float(BLEACH_CFG.get("lora_weight", 0.85))
EXTRA = list(BLEACH_CFG.get("extra_lora_files") or [])
USE_DRIVE = bool(BLEACH_CFG.get("use_drive"))
COMFY = Path(BLEACH_CFG["comfy"])
API = "http://127.0.0.1:8188"

pos_w = w.Textarea(value=POS0, description="", layout=w.Layout(width="100%", height="110px"))
neg_w = w.Textarea(value=NEG0, description="", layout=w.Layout(width="100%", height="70px"))
width_w = w.BoundedIntText(value=832, min=512, max=1280, step=8, description="Rộng:", style={"description_width": "70px"})
height_w = w.BoundedIntText(value=1216, min=512, max=1536, step=8, description="Cao:", style={"description_width": "70px"})
steps_w = w.BoundedIntText(value=28, min=8, max=50, description="Steps:", style={"description_width": "70px"})
cfg_w = w.BoundedFloatText(value=6.0, min=1.0, max=12.0, step=0.5, description="CFG:", style={"description_width": "70px"})
seed_w = w.IntText(value=-1, description="Seed:", style={"description_width": "70px"})
batch_w = w.BoundedIntText(value=1, min=1, max=2, description="Số ảnh:", style={"description_width": "70px"})
btn = w.Button(description="Tạo ảnh", button_style="success", layout=w.Layout(width="200px", height="42px"))
status = w.HTML("")
gallery = w.Output()

def _api_graph(seed, width, height, steps, cfg, batch, pos, neg):
    last = "10"
    g = {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "WAI-Nsfw-Illustrious-15.safetensors"}},
        "10": {"class_type": "LoraLoader", "inputs": {
            "model": ["1", 0], "clip": ["1", 1], "lora_name": "bleach_watercolor.safetensors",
            "strength_model": WEIGHT, "strength_clip": WEIGHT}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {"width": int(width), "height": int(height), "batch_size": int(batch)}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": pos, "clip": [last, 1]}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"text": neg, "clip": [last, 1]}},
        "3": {"class_type": "KSampler", "inputs": {
            "seed": int(seed), "steps": int(steps), "cfg": float(cfg),
            "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0,
            "model": [last, 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["1", 2]}},
        "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": "bleach_orihime", "images": ["8", 0]}},
    }
    prev = "10"
    for i, fname in enumerate(EXTRA):
        nid = str(20 + i)
        g[nid] = {"class_type": "LoraLoader", "inputs": {
            "model": [prev, 0], "clip": [prev, 1], "lora_name": fname,
            "strength_model": WEIGHT, "strength_clip": WEIGHT}}
        prev = nid
    if prev != "10":
        g["6"]["inputs"]["clip"] = [prev, 1]
        g["7"]["inputs"]["clip"] = [prev, 1]
        g["3"]["inputs"]["model"] = [prev, 0]
    return g

def _post(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(API + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode())

def _get(path):
    with urllib.request.urlopen(API + path, timeout=60) as resp:
        return resp.read()

def _run(_=None):
    status.value = "<p>Đang tạo ảnh, đợi chút (lần đầu có thể lâu vì phải nạp model)…</p>"
    gallery.clear_output()
    width, height = int(width_w.value), int(height_w.value)
    if width % 8 or height % 8:
        status.value = '<p style="color:#b00020">Rộng và cao phải chia hết cho 8.</p>'
        return
    seed = int(seed_w.value)
    if seed < 0:
        seed = random.randint(0, 2**32 - 1)
    graph = _api_graph(seed, width, height, int(steps_w.value), float(cfg_w.value),
                       int(batch_w.value), pos_w.value, neg_w.value)
    try:
        queued = _post("/prompt", {"prompt": graph})
    except Exception:
        status.value = '<p style="color:#b00020">Không gửi được tới ComfyUI. Chạy lại ô 2 cho server kịp mở.</p>'
        return
    pid = queued.get("prompt_id")
    if not pid:
        status.value = '<p style="color:#b00020">ComfyUI không nhận prompt.</p>'
        return
    images_meta = []
    for _i in range(300):
        time.sleep(2)
        try:
            hist = json.loads(_get(f"/history/{pid}"))
        except Exception:
            continue
        item = hist.get(pid)
        if not item:
            continue
        if item.get("status", {}).get("completed") or item.get("outputs"):
            for node in (item.get("outputs") or {}).values():
                for im in node.get("images") or []:
                    images_meta.append(im)
            break
    if not images_meta:
        status.value = '<p style="color:#b00020">Hết thời gian chờ hoặc không có ảnh. Thử giảm kích thước xuống 768×1152 rồi tạo lại.</p>'
        return
    saved = []
    with gallery:
        for im in images_meta:
            q = urllib.parse.urlencode({
                "filename": im.get("filename", ""),
                "subfolder": im.get("subfolder", ""),
                "type": im.get("type", "output"),
            })
            raw = _get("/view?" + q)
            out_dir = Path("/tmp/bleach_out")
            out_dir.mkdir(parents=True, exist_ok=True)
            local = out_dir / im.get("filename", "out.png")
            local.write_bytes(raw)
            if USE_DRIVE:
                dest_dir = Path("/content/drive/MyDrive/rpchat_comfyui/output")
                dest_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(local, dest_dir / local.name)
                saved.append(str(dest_dir / local.name))
            display(IPyImage(data=raw))
    msg = f"Xong. Seed = {seed}. {len(images_meta)} ảnh."
    if saved:
        msg += " Đã lưu vào Google Drive/rpchat_comfyui/output."
    status.value = f'<p style="color:#0a7a2f"><b>{msg}</b></p>'

import urllib.parse
btn.on_click(_run)
display(HTML("<p><b>Prompt dương</b></p>"), pos_w, HTML("<p><b>Prompt âm</b></p>"), neg_w)
display(w.HBox([width_w, height_w]), w.HBox([steps_w, cfg_w]), w.HBox([seed_w, batch_w]))
display(HTML("<p>Seed = <code>-1</code> thì mỗi lần một seed ngẫu nhiên.</p>"), btn, status, gallery)


**Gợi ý:** Lần tạo đầu thường lâu vì model ~6,5 GB phải nạp vào GPU. 832×1216 là mức vừa với T4; nếu hết bộ nhớ thì hạ còn 768×1152. Prompt nhân vật dùng tag `inoue orihime` và `bleach_watercolor`. Khi hết phiên Colab, model trên máy ảo bị xóa — bật lưu Google Drive ở ô 1 thì lần sau khỏi tải lại.
